# 📉 Análisis Estratégico de Retención: Telco Customer Churn

**Autor:** David Álvarez  
**Especialidad:** Análisis de Datos & Estrategia Cuantitativa  
**Repositorio:** [github.com/Dalvarez78/telco-churn-sql](https://github.com/Dalvarez78/telco-churn-sql)  

---

## 1. Conexión Relacional a SQLite y Configuración
En este primer paso establezco la conexión con la base de datos relacional `telco_churn.db`, configurada con índices para acelerar las consultas analíticas.

In [1]:
import os
import sqlite3
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

posibles_rutas = [
    Path.cwd() / 'telco_churn.db',
    Path.cwd() / 'data' / 'telco_churn.db',
    Path.cwd() / 'telco-churn-sql' / 'telco_churn.db'
]
ruta_bd = next((r for r in posibles_rutas if r.exists()), Path('telco_churn.db'))
conexion = sqlite3.connect(ruta_bd)
cur = conexion.cursor()
total = cur.execute('SELECT COUNT(*) FROM telco_churn').fetchone()[0]
print(f'✔ Conectado exitosamente a la base de datos relacional: {ruta_bd.name}')
print(f'✔ Registros disponibles: {total:,} clientes.')

def ejecutar_consulta(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conexion)


✔ Conectado exitosamente a la base de datos relacional: telco_churn.db
✔ Registros disponibles: 7,043 clientes.


---
## 2. Consulta 1: Tasa de Abandono y Fuga Mensual de Ingresos (MRR)
Evaluamos dónde se concentra la pérdida recurrente mensual según el tipo de permanencia y el rango de antigüedad (`tenure`).

In [2]:
consulta_1 = """
-- ============================================================================
-- Consulta 1: Tasa de Abandono e Impacto Económico por Contrato y Antigüedad
-- Objetivo: Determinar qué combinación de permanencia contractual y meses
--           de antigüedad concentra la mayor fuga recurrente mensual (MRR).
-- ============================================================================

SELECT 
    Contract AS tipo_contrato,
    CASE 
        WHEN tenure <= 6 THEN '0-6 meses'
        WHEN tenure <= 12 THEN '7-12 meses'
        WHEN tenure <= 24 THEN '1-2 años'
        ELSE 'Más de 2 años'
    END AS rango_antiguedad,
    COUNT(*) AS total_clientes,
    COUNT(CASE WHEN Churn = 'Yes' THEN 1 END) AS clientes_cancelados,
    ROUND(100.0 * COUNT(CASE WHEN Churn = 'Yes' THEN 1 END) / COUNT(*), 1) AS tasa_churn_pct,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS ingresos_mensuales_perdidos,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END) / 
          (SELECT SUM(MonthlyCharges) FROM telco_churn WHERE Churn = 'Yes'), 1) AS pct_del_total_fuga
FROM telco_churn
GROUP BY 1, 2
ORDER BY ingresos_mensuales_perdidos DESC;
"""

resultado_1 = ejecutar_consulta(consulta_1)
resultado_1


tipo_contrato,rango_antiguedad,total_clientes,clientes_cancelados,tasa_churn_pct,ingresos_mensuales_perdidos,pct_del_total_fuga
Month-to-month,0-6 meses,1413,780,55.2,49681.30,35.7
Month-to-month,Más de 2 años,1144,353,30.9,30565.35,22.0
Month-to-month,1-2 años,737,278,37.7,21980.30,15.8
Month-to-month,7-12 meses,581,244,42.0,18620.15,13.4
One year,Más de 2 años,1152,137,11.9,12364.30,8.9
Two year,Más de 2 años,1537,48,3.1,4165.30,3.0
One year,1-2 años,197,16,8.1,1101.35,0.8
One year,7-12 meses,85,9,10.6,438.00,0.3
One year,0-6 meses,39,4,10.3,214.80,0.2
Two year,0-6 meses,29,0,0.0,0.00,0.0


### 💡 Conclusiones de Negocio - Consulta 1:
* **Punto de dolor primario:** El contrato mes a mes (**Month-to-month**) con **menos de 6 meses de antigüedad** sufre un **55.2% de abandono**.
* **Impacto Económico:** Este segmento representa por sí solo una pérdida de **$49,681.30 USD al mes** (el **35.7%** de toda la fuga económica de la empresa).
* **Efecto Blindaje:** Los contratos anuales y bianuales reducen el churn por debajo del 12% y 3% respectivamente.

---
## 3. Consulta 2: Profundización en el Segmento Crítico (0-6 meses sin permanencia)
Analizamos cómo interactúan el tipo de servicio de internet y el método de pago dentro de la cohorte más vulnerable.

In [3]:
consulta_2 = """
-- ============================================================================
-- Consulta 2: Profundización en el Segmento Crítico de Mayor Riesgo
-- Filtro: Contratos mensuales (Month-to-month) con antigüedad <= 6 meses.
-- Objetivo: Identificar qué servicios de internet y métodos de pago impulsan
--           el mayor volumen y porcentaje de cancelación en clientes nuevos.
-- ============================================================================

SELECT 
    InternetService AS servicio_internet,
    PaymentMethod AS metodo_pago,
    COUNT(*) AS total_clientes,
    COUNT(CASE WHEN Churn = 'Yes' THEN 1 END) AS cancelaciones,
    ROUND(100.0 * COUNT(CASE WHEN Churn = 'Yes' THEN 1 END) / COUNT(*), 1) AS tasa_churn_pct,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS mrr_perdido_segmento,
    ROUND(AVG(MonthlyCharges), 2) AS cuota_mensual_promedio
FROM telco_churn
WHERE Contract = 'Month-to-month' 
  AND tenure <= 6
GROUP BY 1, 2
ORDER BY cancelaciones DESC, tasa_churn_pct DESC;
"""

resultado_2 = ejecutar_consulta(consulta_2)
resultado_2


servicio_internet,metodo_pago,total_clientes,cancelaciones,tasa_churn_pct,mrr_perdido_segmento,cuota_mensual_promedio
Fiber optic,Electronic check,440,332,75.5,26920.80,80.80
DSL,Electronic check,209,119,56.9,5213.10,45.63
DSL,Mailed check,192,87,45.3,3964.40,45.97
Fiber optic,Mailed check,96,60,62.5,4731.40,79.36
No,Mailed check,216,57,26.4,1144.85,20.26
Fiber optic,Bank transfer (automatic),41,38,92.7,3098.65,81.59
Fiber optic,Credit card (automatic),42,29,69.0,2399.30,80.81
DSL,Credit card (automatic),52,21,40.4,1006.15,49.71
DSL,Bank transfer (automatic),42,18,42.9,816.00,48.84
No,Electronic check,31,9,29.0,179.35,20.05


### 💡 Conclusiones de Negocio - Consulta 2:
* **Fricción de Pago:** El método `Electronic check` (cheque electrónico) no es automatizado; exige al usuario gestionar el pago mensualmente.
* **Foco Sangrante:** La combinación **Fibra Óptica + Cheque Electrónico** alcanza un **75.5% de abandono** (332 de 440 clientes cancelan), con una pérdida mensual de **$26,920.80 USD**.

---
## 4. Consulta 3: Diagnóstico Causal - Carencia de Soporte Técnico y Seguridad
Evaluamos si la ausencia de asistencia técnica y ciberseguridad actúa como catalizador del abandono temprano.

In [4]:
consulta_3 = """
-- ============================================================================
-- Consulta 3: Diagnóstico Causal - Carencia de Soporte Técnico y Ciberseguridad
-- Filtro: Contratos mensuales (Month-to-month) con antigüedad <= 6 meses.
-- Objetivo: Evaluar si los clientes que no cuentan con soporte técnico y seguridad
--           sufren mayor propensión al abandono, justificando paquetes de bienvenida.
-- ============================================================================

SELECT 
    CASE 
        WHEN TechSupport = 'Yes' THEN 'Con Soporte Técnico'
        WHEN TechSupport = 'No' THEN 'Sin Soporte Técnico'
        ELSE 'Sin Internet'
    END AS estado_soporte_tecnico,
    CASE 
        WHEN OnlineSecurity = 'Yes' THEN 'Con Seguridad Online'
        WHEN OnlineSecurity = 'No' THEN 'Sin Seguridad Online'
        ELSE 'Sin Internet'
    END AS estado_seguridad_online,
    COUNT(*) AS total_clientes,
    COUNT(CASE WHEN Churn = 'Yes' THEN 1 END) AS clientes_cancelados,
    ROUND(100.0 * COUNT(CASE WHEN Churn = 'Yes' THEN 1 END) / COUNT(*), 1) AS tasa_churn_pct,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS mrr_perdido
FROM telco_churn
WHERE Contract = 'Month-to-month' 
  AND tenure <= 6
GROUP BY 1, 2
ORDER BY clientes_cancelados DESC;
"""

resultado_3 = ejecutar_consulta(consulta_3)
resultado_3


estado_soporte_tecnico,estado_seguridad_online,total_clientes,clientes_cancelados,tasa_churn_pct,mrr_perdido
Sin Soporte Técnico,Sin Seguridad Online,860,593,69.0,40638.4
Sin Internet,Sin Internet,299,76,25.4,1531.5
Con Soporte Técnico,Sin Seguridad Online,116,56,48.3,4004.8
Sin Soporte Técnico,Con Seguridad Online,106,42,39.6,2684.5
Con Soporte Técnico,Con Seguridad Online,32,13,40.6,822.1


### 💡 Conclusiones de Negocio - Consulta 3:
* **Causa Operativa:** De los 780 clientes fugados en este segmento, **593 (el 76%) no contaban con Soporte Técnico ni Seguridad Online**.
* Los usuarios sin soporte sufren un **69.0% de churn**, mientras que contar con soporte y seguridad mitiga sustancialmente el abandono.

---
## 5. Consulta 4: Simulación de Negocio, Impacto Financiero y ROI de Retención
Diseñamos una simulación con CTEs para estimar el retorno de inversión de una campaña dirigida a rescatar el 25% de los clientes en riesgo.

In [5]:
consulta_4 = """
-- ============================================================================
-- Consulta 4: Simulación de Negocio, Impacto Financiero y ROI de Retención
-- Objetivo: Proyectar el retorno sobre la inversión (ROI) de la campaña de
--           retención propuesta (15% de descuento durante 6 meses al migrar
--           a contrato de permanencia anual con domiciliación de pago).
-- ============================================================================

WITH metricas_segmento_critico AS (
    SELECT 
        COUNT(*) AS clientes_totales_cohorte,
        COUNT(CASE WHEN Churn = 'Yes' THEN 1 END) AS clientes_fugados,
        SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END) AS mrr_perdido_mensual,
        AVG(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE NULL END) AS cuota_mensual_promedio
    FROM telco_churn
    WHERE Contract = 'Month-to-month' 
      AND tenure <= 6
)
SELECT 
    clientes_fugados AS clientes_perdidos_historicos,
    ROUND(mrr_perdido_mensual, 2) AS mrr_perdido_mensual,
    ROUND(mrr_perdido_mensual * 12, 2) AS fuga_ingresos_anuales,
    
    -- Hipótesis de Retención: Meta alcanzable del 25% de rescate de clientes
    CAST(ROUND(clientes_fugados * 0.25) AS INT) AS meta_clientes_rescatados_25pct,
    ROUND(mrr_perdido_mensual * 0.25 * 12, 2) AS ingreso_anual_protegido,
    
    -- Coste de inversión de la campaña: 15% de descuento durante 6 meses sobre los clientes retenidos
    ROUND((clientes_fugados * 0.25) * (cuota_mensual_promedio * 0.15) * 6, 2) AS coste_inversion_campana,
    
    -- Beneficio Neto estimado para la compañía en el Año 1
    ROUND((mrr_perdido_mensual * 0.25 * 12) - 
          ((clientes_fugados * 0.25) * (cuota_mensual_promedio * 0.15) * 6), 2) AS beneficio_neto_anual_recuperado,
          
    -- Retorno sobre la Inversión (ROI %)
    ROUND((((mrr_perdido_mensual * 0.25 * 12) - 
            ((clientes_fugados * 0.25) * (cuota_mensual_promedio * 0.15) * 6)) / 
           ((clientes_fugados * 0.25) * (cuota_mensual_promedio * 0.15) * 6)) * 100, 1) AS roi_campana_pct
FROM metricas_segmento_critico;
"""

resultado_4 = ejecutar_consulta(consulta_4)
resultado_4


clientes_perdidos_historicos,mrr_perdido_mensual,fuga_ingresos_anuales,meta_clientes_rescatados_25pct,ingreso_anual_protegido,coste_inversion_campana,beneficio_neto_anual_recuperado,roi_campana_pct
780,49681.3,596175.6,195,149043.9,11178.29,137865.61,1233.3


### 💡 Conclusiones Financieras - Simulación de ROI:
* **Fuga Anual en el Segmento Crítico:** $596,175.60 USD / año.
* **Meta Prudente de Rescate (25%):** Salvar a 195 clientes protege **$149,043.90 USD anuales**.
* **Inversión Comercial (15% descuento x 6 meses):** $11,178.29 USD.
* **Beneficio Neto Año 1:** **$137,865.61 USD**.
* **Retorno de Inversión (ROI):** **+1,233.3%**.

---
## 6. Cuadro de Mandos Visual (Data Storytelling)
Visualizaciones ejecutivas generadas a partir de las consultas SQL para sintetizar los hallazgos de negocio.

In [6]:
from IPython.display import Image, display
from pathlib import Path

for g in ['grafico_01_mrr_perdido_contrato.png', 'grafico_02_segmento_riesgo_pago.png', 'grafico_03_soporte_seguridad.png', 'grafico_04_simulacion_roi.png']:
    p = Path('outputs') / g
    if p.exists():
        display(Image(filename=str(p)))


---
## 7. Plan Estratégico de Retención (The Playbook)

1. **Disparador Automatizado (Día 75):** Alerta automática para clientes de fibra óptica sin permanencia que pagan por cheque electrónico.
2. **Incentivo Vinculado:** 15% de descuento durante 6 meses condicionado a la migración a permanencia anual con pago domiciliado automático.
3. **Kit de Onboarding:** 3 meses gratuitos de soporte técnico y seguridad online para reducir la fricción técnica inicial.

---
**Proyecto desarrollado por David Álvarez** — [GitHub (@Dalvarez78)](https://github.com/Dalvarez78/telco-churn-sql)
